# FinAdvisor paper runs on Colab

**How to use:** Runtime → Change runtime type → **T4 GPU**. Then **Runtime → Run all**.

- Every cell checks its own step and is safe to run again. After a crash or a reset, just **Run all** again:
  finished results are on Google Drive and every run continues where it stopped.
- Step 9 runs the quick smoke pass (off by default now that it has passed). Step 10 runs the long paper runs (many
  hours) in the order set in step 1, skipping finished ones.
- If something fails, the cell prints what to send back (the error and the last lines of the Ollama log).

## 1. Settings

In [ ]:
BRANCH = "paper-experiments"
REPO = "https://github.com/b-rahul07/FinAdv.git"
MODELS = ["qwen2.5:3b", "qwen2.5:1.5b", "qwen2.5:7b", "phi4-mini", "llama3.2:3b"]
# Long runs, in this order. Remove ids to skip them; finished personas are skipped automatically.
# Plan-mode runs with the pressure fence on come first: the 11 Nov 2026 fixes (router, number words, the wish in
# twins and repairs) do not change them, so they can finish on the commit they started on. The four runs those fixes
# do change (A6, A1T, A4, ADV) come last and need the fixed code (UPDATE_CODE = True, the default, pulls it).
RUN_ORDER = ["A1_full", "S2_mr6_risk", "A2_no_gate", "A3_no_auditor", "A5_no_calculators", "S1_rules_v1",
             "S1_emergency_3m", "M1_qwen2.5-1.5b", "M1_qwen2.5-7b", "M1_phi4-mini", "M1_llama3.2-3b",
             "B1_plain_llm", "B2_llm_rag", "A6_no_pressure_fence", "A1T_full_turn", "A4_no_number_guard",
             "ADV_adversarial"]
SMOKE_ONLY = ["M1_phi4-mini"]   # configs for the smoke pass; [] = all of them (about an hour)
RUN_SMOKE = False               # step 9 (the smoke pass was done on 9-10 Nov; set True to repeat it)
RUN_LONG = True                 # step 10
UPDATE_CODE = True              # pulls the branch's latest code at step 4; a run never mixes two commits (step 10 checks)

## 2. GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
print("GPU:", gpu.stdout.strip())

## 3. Google Drive (results live here and survive resets)

In [ ]:
import os
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
DRIVE_RESULTS = "/content/drive/MyDrive/finadvisor-results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print("results folder:", DRIVE_RESULTS)

## 4. Code

In [ ]:
import os, subprocess
ROOT = "/content/FinAdv"

def sh(cmd, check=True, quiet=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet and r.stdout.strip():
        print(r.stdout.strip())
    if check and r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise SystemExit(f"failed: {cmd}")
    return r

if not os.path.isdir(f"{ROOT}/.git"):
    sh(f"rm -rf {ROOT} && git clone -q -b {BRANCH} {REPO} {ROOT}")
elif UPDATE_CODE:
    sh(f"git -C {ROOT} pull -q")
os.chdir(ROOT)
link = f"{ROOT}/results"
if os.path.islink(link) and os.readlink(link) != DRIVE_RESULTS:
    os.remove(link)
if os.path.isdir(link) and not os.path.islink(link):
    raise SystemExit(f"{link} is a real folder, not the Drive link; rename it and run again")
if not os.path.exists(link):
    os.symlink(DRIVE_RESULTS, link)
COMMIT = sh("git log --oneline -1", quiet=True).stdout.strip()
COMMIT_HASH = sh("git rev-parse HEAD", quiet=True).stdout.strip()
print("code:", COMMIT)
print("results ->", os.readlink(link))

## 5. Python packages

In [ ]:
import importlib.util
if importlib.util.find_spec("finadvisor") is None or importlib.util.find_spec("chromadb") is None:
    print("installing (about 2 minutes; warnings about Colab's own packages are harmless) ...")
    sh(f"pip install -q -e '{ROOT}[rag]'", quiet=True)
    importlib.invalidate_caches()
print("packages ok")

## 6. Ollama (the model server)

Started with **one model in GPU memory at a time** (`OLLAMA_MAX_LOADED_MODELS=1`). With several models loaded
together the T4 can run out of memory, the likely cause of phi4-mini's error 500.

In [ ]:
import os, shutil, subprocess, time, urllib.request, json
OLLAMA_LOG = "/content/ollama.log"

def ollama_up():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

def ensure_ollama():
    if not shutil.which("ollama"):
        print("installing Ollama ...")
        sh("apt-get install -y -qq zstd", quiet=True)
        sh("curl -fsSL https://ollama.com/install.sh | sh", quiet=True)
    if ollama_up():
        return
    env = {**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_NUM_PARALLEL": "1",
           "OLLAMA_HOST": "127.0.0.1:11434"}
    subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT, env=env,
                     start_new_session=True)
    for _ in range(60):
        if ollama_up():
            print("Ollama is running")
            return
        time.sleep(1)
    print(open(OLLAMA_LOG).read()[-3000:])
    raise SystemExit("Ollama did not start; send the log above")

def ollama_log_tail(n=40):
    try:
        return "".join(open(OLLAMA_LOG).readlines()[-n:])
    except FileNotFoundError:
        return "(no Ollama log)"

ensure_ollama()

## 7. Models (downloaded once per session, about 10 GB in all)

In [ ]:
def pulled():
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10) as r:
        return {m["name"] for m in json.load(r)["models"]}

for m in MODELS:
    have = pulled()
    if m in have or f"{m}:latest" in have:
        print("have", m)
        continue
    print("pulling", m, "...", flush=True)
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(f"could not pull {m}")
print("models:", sorted(pulled()))

## 8. Search index (for B2)

In [ ]:
if not os.path.isdir(f"{ROOT}/data/chroma_db") or not os.listdir(f"{ROOT}/data/chroma_db"):
    print("building the search index (about a minute) ...")
    sh("python scripts/build_index.py", quiet=True)
print("index ok")

## Runner

Runs a command and shows its output live. If it stops (model error, crash), it restarts Ollama and tries again,
up to 3 times; runs resume, so nothing finished is repeated.

In [ ]:
import sys

def stream(cmd):
    p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"})
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail = (tail + [line])[-60:]
    return p.wait(), "".join(tail)

def module_for(run_id):
    if run_id.startswith("B"):
        return "experiments.baselines"
    if run_id.startswith("ADV"):
        return "experiments.adversarial"
    return "experiments.run"

def run_with_retries(cmd, label, tries=3):
    tail = ""
    for attempt in range(1, tries + 1):
        ensure_ollama()
        status, tail = stream(cmd)
        if status == 0:
            return True
        print(f"\n*** {label} stopped (attempt {attempt} of {tries}). Last Ollama log lines:\n{ollama_log_tail(15)}")
        if attempt < tries:
            subprocess.run(["pkill", "-f", "ollama serve"])
            time.sleep(5)
    print(f"\n*** {label} failed. Send this back:\n--- output ---\n{tail}\n--- ollama.log ---\n{ollama_log_tail()}")
    return False

## 9. Smoke pass (1 replicate × 20 personas per config)

In [ ]:
if RUN_SMOKE:
    cmd = [sys.executable, "-m", "experiments.smoke_paper"] + (["--only", *SMOKE_ONLY] if SMOKE_ONLY else [])
    ok = run_with_retries(cmd, "smoke pass", tries=2)
    print("\nSMOKE PASS OK" if ok else "\nSMOKE PASS FAILED: send the output above")
else:
    print("skipped (RUN_SMOKE = False)")

## 10. Long runs (240 personas × 3 replicates; many hours, resumes after any reset)

A run's rows must all come from one commit (the report marks it untraceable otherwise). A run that was started on an
older commit is finished on that commit (the code is checked out, the run completes, the code returns to the latest);
new runs always start on the latest code. The four runs that need the 11 Nov 2026 fixes (A6, A1T, A4, ADV) are never
finished on older code: if one was started there, the cell stops and asks for its file to be deleted.

In [ ]:
import json, yaml

NEED_LATEST = ["A6_no_pressure_fence", "A1T_full_turn", "A4_no_number_guard", "ADV_adversarial"]   # 11 Nov 2026 fixes

def started_on(run_id):
    """(commit of the first row, rows, rows wanted, file) for a run's result file; (None, 0, wanted, file) when empty."""
    cfg = yaml.safe_load(open(f"{ROOT}/experiments/configs/paper/{run_id}.yaml"))
    want = 50 * 3 * cfg.get("per_level", 1) if run_id.startswith("ADV") else 240 * cfg.get("replicates", 1)
    path = f"{ROOT}/{cfg['output']}"
    if not os.path.exists(path):
        return None, 0, want, path
    rows = [l for l in open(path, encoding="utf-8") if l.strip()]
    try:
        first = json.loads(rows[0]).get("git_commit") if rows else None
    except ValueError:
        first = None
    return first, len(rows), want, path

def code_at(commit):
    sh(f"git -C {ROOT} checkout -q {commit}")
    print("code:", sh(f"git -C {ROOT} log --oneline -1", quiet=True).stdout.strip(), flush=True)

if not RUN_LONG:
    print("skipped: set RUN_LONG = True in step 1, then run this cell")
else:
    for run_id in RUN_ORDER:
        commit0, have, want, path = started_on(run_id)
        if have >= want:
            print(f"{run_id}: done ({have}/{want})")
            continue
        pinned = bool(commit0) and commit0 != COMMIT_HASH
        if pinned and run_id in NEED_LATEST:
            print(f"\n*** {run_id} has {have} rows from the older commit {commit0[:10]}; it must run on the latest code. "
                  f"Delete that file and run this cell again:\n    !rm {path}")
            break
        if pinned:
            print(f"\n{run_id}: {have} of {want} rows were made on commit {commit0[:10]}; finishing it on that commit")
            code_at(commit0)
        cfg = f"experiments/configs/paper/{run_id}.yaml"
        print(f"\n========== {run_id} ==========", flush=True)
        ok = run_with_retries([sys.executable, "-m", module_for(run_id), cfg], run_id)
        if pinned:
            code_at(BRANCH)
        if not ok:
            break
    else:
        print("\nALL RUNS FINISHED")

## 11. Progress (run any time)

In [ ]:
import yaml
for run_id in RUN_ORDER:
    cfg = yaml.safe_load(open(f"{ROOT}/experiments/configs/paper/{run_id}.yaml"))
    want = 50 * 3 * cfg.get("per_level", 1) if run_id.startswith("ADV") else 240 * cfg.get("replicates", 1)
    path = f"{ROOT}/{cfg['output']}"
    have = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    print(f"{run_id:22s} {have:4d} / {want}  {'done' if have >= want else ''}")

## 12. Report (when runs are done; writes results/paper/report.md on Drive)

In [ ]:
import glob
files = sorted(glob.glob(f"{ROOT}/results/paper/*.jsonl"))
if files:
    stream([sys.executable, "-m", "experiments.report", *files, "--reference", "A1_full",
            "--out", f"{ROOT}/results/paper/report.md"])
    print(open(f"{ROOT}/results/paper/report.md").read()[:6000])
else:
    print("no results yet")